# Interpretable Deep Neural Framework for Automated Diagnosis of Thyroid Eye Disease (TED)
## Using Facial Image Analysis with Grad-CAM + SHAP Explainability

This notebook:
- Trains a multi-task MobileNetV2 model to detect TED and its symptoms (proptosis, scleral show, chemosis)
- Uses **Grad-CAM** and **SHAP GradientExplainer** for visual interpretability
- Incorporates MTCNN-based face/eye detection for region-specific analysis
- Provides clinically interpretable outputs

## CELL 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q lz4 mtcnn opencv-python shap openpyxl
print('All dependencies installed ✅')

## CELL 2 — Mount Drive & Core Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from mtcnn import MTCNN
import shap

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print('TensorFlow version:', tf.__version__)
print('SHAP version:', shap.__version__)
print('Imports successful ✅')

## CELL 3 — Config & Paths

In [ ]:
# ── Paths ──────────────────────────────────────────────
LABELS_XLSX     = '/content/drive/MyDrive/TED/LABELSDATASET.xlsx'
IMG_DIR         = '/content/drive/MyDrive/TED/Images'
MODEL_SAVE_PATH = '/content/drive/MyDrive/TED/ted_multitask_model.keras'  # Native Keras format (recommended)
BEST_CKPT_PATH  = '/content/best_multitask.keras'                          # Native Keras format (recommended)

# ── Hyperparams ────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH      = 16
EPOCHS     = 25
SEED       = 42

print('Excel exists :', os.path.exists(LABELS_XLSX))
print('Images folder:', os.path.exists(IMG_DIR))

## CELL 4 — Load & Validate Labels

In [ ]:
df = pd.read_excel(LABELS_XLSX)
df.columns = df.columns.str.strip()

print('Rows, Cols:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
IMG_COL = 'Image'
Y_COLS  = ['Proptosis', 'Scleral show', 'Chemosis', 'TED']

# Normalize filenames
df[IMG_COL] = df[IMG_COL].astype(str).str.strip()
df[IMG_COL] = df[IMG_COL].str.replace(r'\.(jpeg|png)$', '.jpg', regex=True, case=False)
df['img_path'] = df[IMG_COL].apply(lambda x: os.path.join(IMG_DIR, x))

before = len(df)
df = df[df['img_path'].apply(os.path.exists)].copy()
after  = len(df)
print(f'Rows before: {before}, after removing missing files: {after}')
print(f'Missing removed: {before - after}')

Y         = df[Y_COLS].astype('float32').values
img_paths = df['img_path'].values

ted_pos = int(Y[:, 3].sum())
print(f'TED positives: {ted_pos} / {len(Y)}')

## CELL 5 — Train / Val / Test Split

In [ ]:
strat = Y[:, 3].astype(int)
idx   = np.arange(len(df))

train_idx, test_idx = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=strat)
train_idx, val_idx  = train_test_split(train_idx, test_size=0.15, random_state=SEED, stratify=strat[train_idx])

train_paths, train_Y = img_paths[train_idx], Y[train_idx]
val_paths,   val_Y   = img_paths[val_idx],   Y[val_idx]
test_paths,  test_Y  = img_paths[test_idx],  Y[test_idx]

print(f'Train / Val / Test: {len(train_paths)} / {len(val_paths)} / {len(test_paths)}')
print(f'Train TED positives: {int(train_Y[:, 3].sum())} / {len(train_Y)}')

## CELL 6 — Image Loaders & tf.data Pipelines

In [ ]:
def load_image(path):
    """Standard 224x224 loader (used for training & SHAP background)."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img


def load_image_no_crop(path):
    """Letterbox loader — preserves aspect ratio, pads remaining area."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)

    target_h, target_w = IMG_SIZE
    h = tf.cast(tf.shape(img)[0], tf.float32)
    w = tf.cast(tf.shape(img)[1], tf.float32)
    scale  = tf.minimum(target_h / h, target_w / w)
    new_h  = tf.cast(tf.round(h * scale), tf.int32)
    new_w  = tf.cast(tf.round(w * scale), tf.int32)
    img    = tf.image.resize(img, (new_h, new_w))

    pad_h     = target_h - new_h
    pad_w     = target_w - new_w
    pad_top   = pad_h // 2
    pad_bot   = pad_h - pad_top
    pad_left  = pad_w // 2
    pad_right = pad_w - pad_left
    img = tf.pad(img, [[pad_top, pad_bot], [pad_left, pad_right], [0, 0]], constant_values=0.0)
    img = tf.ensure_shape(img, (target_h, target_w, 3))
    return img


def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.12)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 1.0)


def make_ds(paths, Y_arr, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, Y_arr))

    def _map(path, y):
        img = load_image(path)
        if training:
            img = augment(img)
        return img, {
            'proptosis': y[0],
            'scleral':   y[1],
            'chemosis':  y[2],
            'ted':       y[3],
        }

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(512, seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(train_paths, train_Y, training=True)
val_ds   = make_ds(val_paths,   val_Y,   training=False)
test_ds  = make_ds(test_paths,  test_Y,  training=False)

for x, y in train_ds.take(1):
    print('Batch image shape:', x.shape)
    print('TED label shape  :', y['ted'].shape)
print('Datasets built ✅')

## CELL 7 — Build Multi-Task Model

In [ ]:
inp  = keras.Input(shape=(224, 224, 3), name='image')
base = keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_tensor=inp
)
base.trainable = False  # freeze backbone for small dataset

x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

out_proptosis = layers.Dense(1, activation='sigmoid', name='proptosis')(x)
out_scleral   = layers.Dense(1, activation='sigmoid', name='scleral')(x)
out_chemosis  = layers.Dense(1, activation='sigmoid', name='chemosis')(x)
out_ted       = layers.Dense(1, activation='sigmoid', name='ted')(x)

model = keras.Model(
    inputs=inp,
    outputs=[out_proptosis, out_scleral, out_chemosis, out_ted]
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={
        'proptosis': 'binary_crossentropy',
        'scleral':   'binary_crossentropy',
        'chemosis':  'binary_crossentropy',
        'ted':       'binary_crossentropy',
    },
    metrics={
        'proptosis': [keras.metrics.AUC(name='auc')],
        'scleral':   [keras.metrics.AUC(name='auc')],
        'chemosis':  [keras.metrics.AUC(name='auc')],
        'ted':       [
            keras.metrics.AUC(name='auc'),
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ],
    }
)

model.summary()

## CELL 8 — Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True,
        monitor='val_ted_auc',
        mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.5,
        monitor='val_ted_auc',
        mode='max'
    ),
    # ✅ Native Keras format — no more HDF5 legacy warning
    keras.callbacks.ModelCheckpoint(
        BEST_CKPT_PATH,
        save_best_only=True,
        monitor='val_ted_auc',
        mode='max',
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## CELL 9 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# TED AUC
axes[0].plot(history.history.get('ted_auc', []), label='Train TED AUC')
axes[0].plot(history.history.get('val_ted_auc', []), label='Val TED AUC')
axes[0].set_title('TED AUC over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].legend()
axes[0].grid(True)

# Total Loss
axes[1].plot(history.history.get('loss', []), label='Train Loss')
axes[1].plot(history.history.get('val_loss', []), label='Val Loss')
axes[1].set_title('Total Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## CELL 10 — Evaluation on Test Set

In [ ]:
# Load best checkpoint weights
model.load_weights(BEST_CKPT_PATH)

pred_propt, pred_scleral, pred_chemosis, pred_ted = model.predict(test_ds, verbose=1)

# ── TED evaluation ────────────────────────────────────────────────────
pred_ted_prob = pred_ted.ravel()
pred_ted_cls  = (pred_ted_prob >= 0.5).astype(int)
true_ted      = test_Y[:, 3].astype(int)

print('\n' + '='*50)
print('TED EVALUATION')
print('='*50)
print('Confusion Matrix:')
print(confusion_matrix(true_ted, pred_ted_cls))
print(classification_report(true_ted, pred_ted_cls, digits=4, zero_division=0))
print('ROC-AUC:', roc_auc_score(true_ted, pred_ted_prob))

# ── Symptom evaluation ────────────────────────────────────────────────
preds_list = [pred_propt, pred_scleral, pred_chemosis]
names_list = ['Proptosis', 'Scleral show', 'Chemosis']

for i, name in enumerate(names_list):
    prob = preds_list[i].ravel()
    cls  = (prob >= 0.5).astype(int)
    true = test_Y[:, i].astype(int)
    print(f'\n== {name} ==')
    print(confusion_matrix(true, cls))
    # zero_division=0 silences the UndefinedMetricWarning cleanly
    print(classification_report(true, cls, digits=4, zero_division=0))

## CELL 11 — Save Model (Native Keras Format)

In [ ]:
# ✅ Save in native .keras format (recommended over legacy .h5)
model.save(MODEL_SAVE_PATH)
print('Model saved to:', MODEL_SAVE_PATH)

---
# EXPLAINABILITY: Grad-CAM + SHAP

## CELL 12 — Load Saved Model (skip if training session is still active)

In [ ]:
# Only run this cell if starting a fresh Colab session after training
# Otherwise the model object from CELL 7-8 is already in memory
import os
if 'model' not in dir():
    model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    print('Model loaded from Drive ✅')
else:
    print('Model already in memory ✅')

## CELL 13 — MTCNN Setup & Iris Mask Utility

In [ ]:
detector = MTCNN()


def iris_mask_224(img_path, radius_scale=0.35):
    """Returns a binary mask (224x224) covering the iris/pupil regions.
    Used to suppress SHAP attention inside the iris (artifact suppression).
    """
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return np.zeros((224, 224), dtype=np.uint8)

    h0, w0  = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    faces   = detector.detect_faces(img_rgb)
    if not faces:
        return np.zeros((224, 224), dtype=np.uint8)

    face = sorted(faces, key=lambda x: x.get('confidence', 0), reverse=True)[0]
    k    = face.get('keypoints', {})
    if 'left_eye' not in k or 'right_eye' not in k:
        return np.zeros((224, 224), dtype=np.uint8)

    (lx, ly) = k['left_eye']
    (rx, ry) = k['right_eye']

    # Scale to 224x224 coordinate space
    sx, sy   = 224.0 / w0, 224.0 / h0
    lx, ly   = int(lx * sx), int(ly * sy)
    rx, ry   = int(rx * sx), int(ry * sy)
    eye_dist = np.sqrt((lx - rx)**2 + (ly - ry)**2)
    r        = max(int(eye_dist * radius_scale), 8)

    mask = np.zeros((224, 224), dtype=np.uint8)
    cv2.circle(mask, (lx, ly), r, 1, -1)
    cv2.circle(mask, (rx, ry), r, 1, -1)
    return mask


print('MTCNN detector ready ✅')

## CELL 14 — Grad-CAM Implementation

In [ ]:
def pick_best_conv_layer(model):
    """Select the best intermediate conv layer for Grad-CAM (higher-res than final layer)."""
    preferred = [
        'block_13_expand_relu',   # 14×14 feature map
        'block_6_expand_relu',    # 28×28 feature map
        'block_12_expand_relu',
        'block_16_project',
        'Conv_1'
    ]
    layer_names = [l.name for l in model.layers]
    for n in preferred:
        if n in layer_names:
            return n
    # Fallback: last Conv2D
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError('No suitable Conv2D layer found in model.')


LAST_CONV_LAYER = pick_best_conv_layer(model)
print(f'Grad-CAM conv layer: {LAST_CONV_LAYER}')


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, output_index=3):
    """Compute Grad-CAM heatmap for a given output head.

    Args:
        img_array         : shape (1, H, W, 3), float32 in [0,1]
        model             : compiled Keras model
        last_conv_layer_name: name of the conv layer to tap
        output_index      : 0=proptosis, 1=scleral, 2=chemosis, 3=ted

    Returns:
        heatmap (np.ndarray, shape = conv output spatial dims)
    """
    last_conv = model.get_layer(last_conv_layer_name)

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[last_conv.output, model.outputs]
    )

    with tf.GradientTape() as tape:
        conv_out, outputs = grad_model(img_array, training=False)
        tape.watch(conv_out)
        loss = outputs[output_index][:, 0]

    grads       = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap     = tf.reduce_sum(conv_out[0] * pooled_grads, axis=-1)
    heatmap     = tf.maximum(heatmap, 0)
    heatmap     = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_eye_gradcam(img_path, output_name='ted', output_index=3, alpha=0.55, clip_percent=75):
    """Display Grad-CAM overlay for a given output head.
    Skips heatmap rendering if TED probability < 0.6.
    """
    img       = load_image_no_crop(img_path)
    img_array = tf.expand_dims(img, 0)

    p     = model.predict(img_array, verbose=0)
    probs = {
        'proptosis': float(p[0][0][0]),
        'scleral':   float(p[1][0][0]),
        'chemosis':  float(p[2][0][0]),
        'ted':       float(p[3][0][0]),
    }
    print('Predicted probabilities:', probs)

    if probs['ted'] < 0.6 and output_name == 'ted':
        print(f"TED probability ({probs['ted']:.3f}) < 0.6 → heatmaps skipped.")
        plt.figure(figsize=(4, 4))
        plt.imshow(img.numpy())
        plt.title('Input (Heatmaps skipped — low TED probability)')
        plt.axis('off')
        plt.show()
        return

    heatmap = make_gradcam_heatmap(img_array, model, LAST_CONV_LAYER, output_index=output_index)

    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], IMG_SIZE)
    heatmap_resized = tf.squeeze(heatmap_resized).numpy()

    thr             = np.percentile(heatmap_resized, clip_percent)
    heatmap_resized = np.where(heatmap_resized >= thr, heatmap_resized, 0)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img.numpy())
    axes[0].set_title('Input (letterbox)')
    axes[0].axis('off')

    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title(f'Grad-CAM — {output_name}')
    axes[1].axis('off')

    axes[2].imshow(img.numpy())
    axes[2].imshow(heatmap_resized, cmap='jet', alpha=alpha)
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


print('Grad-CAM utilities ready ✅')

## CELL 15 — SHAP: Build TED Sub-model & Background Dataset

> **Why a sub-model?** `shap.GradientExplainer` requires a single-output model. We extract the TED head only.

In [ ]:
# ── TED-only sub-model (single output) ──────────────────────────────
ted_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.outputs[3],   # index 3 = 'ted' head
    name='ted_submodel'
)
print('TED sub-model output shape:', ted_model.output_shape)

# ── Background set (representative sample from training images) ──────
# SHAP GradientExplainer needs a reference distribution; 20-50 images is sufficient.
N_BACKGROUND   = min(20, len(train_paths))
bg_paths       = random.sample(list(train_paths), N_BACKGROUND)
background     = np.stack(
    [load_image(p).numpy() for p in bg_paths],
    axis=0
).astype(np.float32)
print(f'Background shape: {background.shape}  ({N_BACKGROUND} images from training set)')

# ── Instantiate SHAP GradientExplainer ──────────────────────────────
# GradientExplainer is the correct choice for CNNs (vs DeepExplainer which
# has known issues with BatchNorm layers used in MobileNetV2).
explainer = shap.GradientExplainer(ted_model, background)
print('SHAP GradientExplainer ready ✅')

## CELL 16 — SHAP: User-Friendly Explanation Function

In [ ]:
def user_friendly_ted_explanation(
    img_path,
    ted_thresh=0.50,
    ignore_iris=True,
    radius_scale=0.35,
    clip_percent=90,
    grid=14,
    topk=4
):
    """
    Full SHAP-based explanation for a single image.

    Parameters
    ----------
    img_path     : path to the input image
    ted_thresh   : probability threshold for TED positive decision
    ignore_iris  : if True, suppress SHAP attention inside iris (reduces artifact)
    radius_scale : iris circle radius relative to inter-eye distance
    clip_percent : percentile below which SHAP values are zeroed (focus on top regions)
    grid         : number of grid cells per axis for region summary
    topk         : number of top evidence regions to report
    """
    # ── Load & preprocess image ──────────────────────────────────────
    x  = load_image(img_path).numpy().astype(np.float32)   # (224,224,3)
    xb = np.expand_dims(x, axis=0)                          # (1,224,224,3)

    # ── TED probability ──────────────────────────────────────────────
    ted_prob = float(ted_model.predict(xb, verbose=0)[0][0])

    # ── SHAP values ──────────────────────────────────────────────────
    # GradientExplainer.shap_values returns a list-of-arrays when the
    # model has multiple outputs, or a single array for single output.
    shap_vals = explainer.shap_values(xb)      # list or array

    # Normalise to always get shape (1, 224, 224, 3)
    if isinstance(shap_vals, list):
        # Multi-output case (shouldn't happen with ted_model, but guard anyway)
        sv = np.array(shap_vals[0])            # (1,224,224,3) or (224,224,3)
    else:
        sv = np.array(shap_vals)               # (1,224,224,3)

    sv = np.squeeze(sv)                        # (224,224,3)

    if sv.ndim == 2:                           # edge case: (224,224)
        sv = np.stack([sv] * 3, axis=-1)
    if sv.shape[-1] != 3:                      # edge case: wrong channel count
        sv = np.repeat(sv[..., :1], 3, axis=-1)

    # ── Positive evidence map ────────────────────────────────────────
    # Sum across colour channels; keep only positive (pro-TED) contributions
    shap_map2d = sv.sum(axis=-1)               # (224,224)
    pos        = np.maximum(shap_map2d, 0)

    # Suppress iris/pupil region (high-contrast iris dominates spuriously)
    if ignore_iris:
        iris_m = iris_mask_224(img_path, radius_scale=radius_scale)  # (224,224) binary
        pos    = np.where(iris_m == 1, 0, pos)

    # Normalise & clip to top activations
    pos       = pos / (np.max(pos) + 1e-8)
    thr       = np.percentile(pos, clip_percent)
    pos_clean = np.where(pos >= thr, pos, 0)

    # ── Region summary ───────────────────────────────────────────────
    H, W     = pos.shape
    gh = gw  = grid
    cell_h   = H // gh
    cell_w   = W // gw

    region_scores = []
    for i in range(gh):
        for j in range(gw):
            y1, y2 = i * cell_h, (i + 1) * cell_h
            x1, x2 = j * cell_w, (j + 1) * cell_w
            region_scores.append((float(pos[y1:y2, x1:x2].mean()), i, j))

    region_scores.sort(reverse=True, key=lambda t: t[0])
    top_regions = region_scores[:topk]

    # ── Evidence strength ────────────────────────────────────────────
    evidence_strength = float(pos.mean())
    has_ted           = ted_prob >= ted_thresh

    if   ted_prob >= 0.80: conf_word = 'high'
    elif ted_prob >= 0.60: conf_word = 'moderate'
    elif ted_prob >= 0.40: conf_word = 'low'
    else:                  conf_word = 'very low'

    # ── Print explanation ────────────────────────────────────────────
    print('\n' + '='*50)
    print('USER EXPLANATION (SHAP-based)')
    print('='*50)
    print(f'TED probability : {ted_prob*100:.1f}%  ({conf_word} confidence)')

    if has_ted:
        print('\nResult  : TED is LIKELY.')
        print('Reason  : Abnormal peri-orbital patterns consistent with TED were detected.')
    else:
        print('\nResult  : TED is UNLIKELY.')
        print('Reason  : No strong abnormal patterns detected around the eye region.')

    strength_label = (
        'very weak'  if evidence_strength < 0.03 else
        'mild'       if evidence_strength < 0.07 else
        'noticeable'
    )
    print(f'\nEvidence strength : {strength_label} ({evidence_strength:.4f})')

    print('\nTop evidence regions (14×14 grid):')
    for rank, (score, i, j) in enumerate(top_regions, 1):
        if score > 0:
            print(f'  {rank}) Grid cell ({i},{j})  — score {score:.4f}')

    print('\n⚠  Clinical note:')
    print('   If you experience bulging eyes, redness, swelling, pain, or double vision,')
    print('   please consult an ophthalmologist. This tool is for research purposes only.')

    # ── Visualization ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(x)
    axes[0].set_title('Input Image (224×224)')
    axes[0].axis('off')

    axes[1].imshow(pos_clean, cmap='hot')
    axes[1].set_title('SHAP Evidence Map\n(positive contributions for TED)')
    axes[1].axis('off')

    axes[2].imshow(x)
    axes[2].imshow(pos_clean, cmap='jet', alpha=0.50)
    axes[2].set_title(f'SHAP Overlay\nTED prob: {ted_prob:.2%}')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    return {
        'ted_prob':          ted_prob,
        'has_ted':           int(has_ted),
        'evidence_strength': evidence_strength,
        'conf_word':         conf_word,
    }


print('SHAP explanation function ready ✅')

## CELL 17 — Upload an Image & Run Full Analysis (Grad-CAM + SHAP)

In [ ]:
from google.colab import files

uploaded = files.upload()
img_name = next(iter(uploaded.keys()))
img_path = '/content/' + img_name
print(f'Image uploaded: {img_path}')

# ── 1. Quick probability prediction ─────────────────────────────────
x       = load_image(img_path)
x_batch = tf.expand_dims(x, 0)
p_propt, p_scl, p_che, p_ted = model.predict(x_batch, verbose=0)

print(f'\nProptosis   : {float(p_propt[0][0]):.4f}')
print(f'Scleral show: {float(p_scl[0][0]):.4f}')
print(f'Chemosis    : {float(p_che[0][0]):.4f}')
print(f'TED         : {float(p_ted[0][0]):.4f}')

# Rule-based confirmation (≥2 symptoms → TED)
votes    = sum([
    int(p_propt[0][0] >= 0.5),
    int(p_scl[0][0]   >= 0.5),
    int(p_che[0][0]   >= 0.5)
])
rule_ted = 1 if votes >= 2 else 0
print(f'Rule-based TED (≥2 symptoms positive): {rule_ted}')

In [ ]:
# ── 2. Grad-CAM for all four outputs ──────────────────────────────────
print('\n--- Grad-CAM: TED ---')
show_eye_gradcam(img_path, output_name='ted',       output_index=3)

print('\n--- Grad-CAM: Proptosis ---')
show_eye_gradcam(img_path, output_name='proptosis', output_index=0)

print('\n--- Grad-CAM: Scleral show ---')
show_eye_gradcam(img_path, output_name='scleral',   output_index=1)

print('\n--- Grad-CAM: Chemosis ---')
show_eye_gradcam(img_path, output_name='chemosis',  output_index=2)

In [ ]:
# ── 3. SHAP-based user explanation ──────────────────────────────────
result = user_friendly_ted_explanation(
    img_path,
    ted_thresh=0.50,
    ignore_iris=True
)
print('\nReturned result dict:', result)

## CELL 18 — (Optional) Batch SHAP on Test Set

Compute SHAP values for the full test set and visualise the mean absolute attribution map.

In [ ]:
# Load test images as numpy array
test_imgs = np.stack(
    [load_image(p).numpy() for p in test_paths],
    axis=0
).astype(np.float32)
print(f'Test image array: {test_imgs.shape}')

# Compute SHAP values for the whole test set
# ranked_outputs=1 ensures we only get values for the top output (TED)
shap_vals_test = explainer.shap_values(test_imgs, ranked_outputs=1)

# shap_vals_test is a list of length 1 (one output) containing an array of shape (N,224,224,3)
sv_arr = np.array(shap_vals_test[0])        # (N,224,224,3)

# Mean absolute attribution map (tells us WHERE the model looks on average)
mean_abs_map = np.abs(sv_arr).mean(axis=(0, -1))   # (224,224)

plt.figure(figsize=(6, 5))
plt.imshow(mean_abs_map, cmap='hot')
plt.colorbar(label='Mean |SHAP|')
plt.title('Mean Absolute SHAP Attribution\n(Test Set — TED head)')
plt.axis('off')
plt.tight_layout()
plt.show()

print('Batch SHAP analysis complete ✅')